In [ ]:
%%sql -r dataframe_11
USE WAREHOUSE BIGDATA_MZMB_WH;
USE DATABASE BIGDATA_TAXI_MZMB;
USE SCHEMA GOLD;

CREATE OR REPLACE TABLE GOLD.T7_T10_XGB_RESULTS (
    run_id STRING,
    task STRING,
    model_name STRING,
    model_scope STRING,
    dataset STRING,
    feature_set STRING,
    compute_type STRING,
    scaling_label STRING,

    train_split STRING,
    trainer_eval_split STRING,
    metric_split STRING,

    train_sample_pct FLOAT,
    trainer_eval_sample_pct FLOAT,
    metric_sample_pct FLOAT,

    train_rows NUMBER,
    trainer_eval_rows NUMBER,
    metric_rows NUMBER,

    n_estimators NUMBER,
    train_seconds FLOAT,
    predict_seconds FLOAT,

    mae FLOAT,
    rmse FLOAT,
    r2 FLOAT,

    sum_abs_error FLOAT,
    sum_sq_error FLOAT,
    sum_y FLOAT,
    sum_y2 FLOAT,

    peak_process_memory_mb FLOAT,
    peak_gpu_memory_mb FLOAT,

    ray_nodes NUMBER,
    ray_resources STRING,

    warehouse_name STRING,
    warehouse_size STRING,

    notes STRING,
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE GOLD.T7_T10_XGB_FEATURE_IMPORTANCE (
    run_id STRING,
    model_name STRING,
    feature_set STRING,
    feature_name STRING,
    importance_type STRING,
    importance_value FLOAT,
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
import os
import time
import uuid
import math
import threading
import subprocess

import numpy as np
import pandas as pd
import xgboost as xgb

from snowflake.snowpark.context import get_active_session
from snowflake.ml.data.data_connector import DataConnector
from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig

session = get_active_session()

session.sql("""
    SELECT
        CURRENT_ROLE() AS role,
        CURRENT_WAREHOUSE() AS warehouse,
        CURRENT_DATABASE() AS database_name,
        CURRENT_SCHEMA() AS schema_name
""").show()

In [ ]:
import ray
from snowflake.ml.runtime_cluster import scale_cluster, get_nodes, get_ray_dashboard_url

ray.init(address="auto", ignore_reinit_error=True)

def print_ray_status():
    print("Current Ray nodes:", len(get_nodes()))
    print("Ray cluster resources:", ray.cluster_resources())
    print("Ray available resources:", ray.available_resources())

    try:
        print("Ray dashboard:", get_ray_dashboard_url())
    except Exception as e:
        print("Dashboard unavailable:", e)


def scale_ray_cluster_to(n_nodes):
    print(f"Scaling Ray cluster to {n_nodes} node(s)...")

    scale_cluster(
        expected_cluster_size=n_nodes,
        options={
            "rollback_after_seconds": 1800,
            "block_until_min_cluster_size": n_nodes,
        }
    )

    print_ray_status()


print_ray_status()

In [ ]:
TABLE = "GOLD.T7_DEMAND_FEATURES"

TARGET_LOG = "LOG_TRIP_COUNT"
TARGET_RAW = "TRIP_COUNT"

BASELINE_FEATURES = [
    "DATASET_ID",
    "PICKUP_LOCATION_ID",
    "HOUR_SIN",
    "HOUR_COS",
    "DOW_SIN",
    "DOW_COS",
    "MONTH_SIN",
    "MONTH_COS",
    "IS_WEEKEND",
    "IS_RUSH_HOUR",
    "IS_NIGHT",
    "LAG_1H_TRIP_COUNT_FILLED",
    "LAG_24H_TRIP_COUNT_FILLED",
    "LAG_168H_TRIP_COUNT_FILLED",
    "ROLLING_24H_AVG_TRIP_COUNT_FILLED",
    "ROLLING_168H_AVG_TRIP_COUNT_FILLED",
]

AUGMENTED_FEATURES = BASELINE_FEATURES + [
    "BOROUGH_ID",
    "SERVICE_ZONE_ID",
    "TEMP_MAX_C",
    "TEMP_MIN_C",
    "PRECIPITATION_MM",
    "WEATHER_MISSING",
    "SCHOOL_COUNT",
    "ATTRACTION_COUNT",
    "ACTIVE_EVENTS",
]

In [ ]:
def get_process_memory_mb():
    try:
        import psutil
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / 1024**2
    except Exception:
        return None


def get_gpu_memory_mb():
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used",
                "--format=csv,noheader,nounits"
            ],
            capture_output=True,
            text=True,
            timeout=5
        )

        values = []
        for line in result.stdout.strip().splitlines():
            line = line.strip()
            if line:
                values.append(float(line))

        return max(values) if values else None

    except Exception:
        return None


class ResourceMonitor:
    def __init__(self, interval=1.0):
        self.interval = interval
        self.running = False
        self.peak_process_memory_mb = 0.0
        self.peak_gpu_memory_mb = 0.0
        self.thread = None

    def _monitor(self):
        while self.running:
            mem = get_process_memory_mb()
            gpu = get_gpu_memory_mb()

            if mem is not None:
                self.peak_process_memory_mb = max(self.peak_process_memory_mb, mem)

            if gpu is not None:
                self.peak_gpu_memory_mb = max(self.peak_gpu_memory_mb, gpu)

            time.sleep(self.interval)

    def __enter__(self):
        self.running = True
        self.thread = threading.Thread(target=self._monitor, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.running = False
        if self.thread:
            self.thread.join(timeout=2)

In [ ]:
RESULT_COLUMNS_NO_CREATED = [
    "RUN_ID",
    "TASK",
    "MODEL_NAME",
    "MODEL_SCOPE",
    "DATASET",
    "FEATURE_SET",
    "COMPUTE_TYPE",
    "SCALING_LABEL",

    "TRAIN_SPLIT",
    "TRAINER_EVAL_SPLIT",
    "METRIC_SPLIT",

    "TRAIN_SAMPLE_PCT",
    "TRAINER_EVAL_SAMPLE_PCT",
    "METRIC_SAMPLE_PCT",

    "TRAIN_ROWS",
    "TRAINER_EVAL_ROWS",
    "METRIC_ROWS",

    "N_ESTIMATORS",
    "TRAIN_SECONDS",
    "PREDICT_SECONDS",

    "MAE",
    "RMSE",
    "R2",

    "SUM_ABS_ERROR",
    "SUM_SQ_ERROR",
    "SUM_Y",
    "SUM_Y2",

    "PEAK_PROCESS_MEMORY_MB",
    "PEAK_GPU_MEMORY_MB",

    "RAY_NODES",
    "RAY_RESOURCES",

    "WAREHOUSE_NAME",
    "WAREHOUSE_SIZE",

    "NOTES",
]


def sql_value(value):
    if value is None:
        return "NULL"

    if isinstance(value, (float, np.floating)):
        if math.isnan(float(value)) or math.isinf(float(value)):
            return "NULL"
        return str(float(value))

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    text = str(value)
    text = text.replace("'", "''")
    return f"'{text}'"


def save_xgb_result(result):
    cols = ", ".join(RESULT_COLUMNS_NO_CREATED)
    vals = ", ".join(sql_value(result.get(c)) for c in RESULT_COLUMNS_NO_CREATED)

    session.sql(f"""
        INSERT INTO GOLD.T7_T10_XGB_RESULTS ({cols})
        VALUES ({vals})
    """).collect()


def save_feature_importance(run_id, model_name, feature_set, booster):
    importance = booster.get_score(importance_type="gain")

    rows = []
    for feature_name, value in importance.items():
        rows.append({
            "RUN_ID": run_id,
            "MODEL_NAME": model_name,
            "FEATURE_SET": feature_set,
            "FEATURE_NAME": feature_name,
            "IMPORTANCE_TYPE": "gain",
            "IMPORTANCE_VALUE": float(value),
        })

    if rows:
        session.create_dataframe(rows).write.mode("append").save_as_table(
            "GOLD.T7_T10_XGB_FEATURE_IMPORTANCE"
        )

In [ ]:
def get_warehouse_info():
    warehouse_name = session.sql("SELECT CURRENT_WAREHOUSE()").collect()[0][0]

    warehouse_size = None
    try:
        rows = session.sql(f"SHOW WAREHOUSES LIKE '{warehouse_name}'").collect()
        if rows:
            d = rows[0].as_dict()
            warehouse_size = d.get("size") or d.get("SIZE")
    except Exception:
        pass

    return warehouse_name, warehouse_size


def get_ray_metadata():
    try:
        ray_nodes = len(get_nodes())
    except Exception:
        ray_nodes = None

    try:
        ray_resources = str(ray.cluster_resources())
    except Exception:
        ray_resources = None

    return ray_nodes, ray_resources

In [ ]:
def sample_condition_sql(sample_pct):
    if sample_pct is None or sample_pct >= 100:
        return ""

    threshold = int(sample_pct * 100)  # 1% -> 100 out of 10000

    return f"""
        AND MOD(ABS(HASH(DATASET, PICKUP_LOCATION_ID, PICKUP_HOUR)), 10000) < {threshold}
    """


def dataset_filter_sql(dataset):
    if dataset is None:
        return ""
    return f"AND DATASET = '{dataset}'"


def make_train_sdf(features, dataset=None, train_sample_pct=100.0):
    cols = features + [TARGET_LOG]

    sql = f"""
        SELECT {", ".join(cols)}
        FROM {TABLE}
        WHERE SPLIT = 'train'
        {dataset_filter_sql(dataset)}
        {sample_condition_sql(train_sample_pct)}
    """

    return session.sql(sql)


def make_trainer_eval_sdf(features, dataset=None, trainer_eval_sample_pct=1.0):
    cols = features + [TARGET_LOG]

    sql = f"""
        SELECT {", ".join(cols)}
        FROM {TABLE}
        WHERE SPLIT = 'validation'
        {dataset_filter_sql(dataset)}
        {sample_condition_sql(trainer_eval_sample_pct)}
    """

    return session.sql(sql)


def make_metric_sdf(features, split="test", dataset=None, metric_sample_pct=100.0):
    cols = features + [TARGET_LOG, TARGET_RAW]

    sql = f"""
        SELECT {", ".join(cols)}
        FROM {TABLE}
        WHERE SPLIT = '{split}'
        {dataset_filter_sql(dataset)}
        {sample_condition_sql(metric_sample_pct)}
    """

    return session.sql(sql)

In [ ]:
def compute_metrics_from_sums(n, sum_abs_error, sum_sq_error, sum_y, sum_y2):
    mae = sum_abs_error / n
    rmse = (sum_sq_error / n) ** 0.5

    y_mean = sum_y / n
    ss_tot = sum_y2 - n * (y_mean ** 2)

    r2 = 1.0 - (sum_sq_error / ss_tot) if ss_tot > 0 else None

    return mae, rmse, r2


def evaluate_xgb_batched(
    booster,
    features,
    split="test",
    dataset=None,
    metric_sample_pct=100.0
):
    sdf = make_metric_sdf(
        features=features,
        split=split,
        dataset=dataset,
        metric_sample_pct=metric_sample_pct
    )

    n = 0
    sum_abs_error = 0.0
    sum_sq_error = 0.0
    sum_y = 0.0
    sum_y2 = 0.0

    start = time.time()

    for pdf in sdf.to_pandas_batches():
        X_df = pdf[features].astype("float32")
        y_true = pdf[TARGET_RAW].astype("float64").values

        dmatrix = xgb.DMatrix(X_df, feature_names=features)
        pred_log = booster.predict(dmatrix)

        pred_log = np.clip(pred_log, 0, np.log1p(100000))
        pred_raw = np.expm1(pred_log)
        pred_raw = np.clip(pred_raw, 0, None)

        errors = y_true - pred_raw

        n_batch = len(y_true)
        n += n_batch
        sum_abs_error += float(np.abs(errors).sum())
        sum_sq_error += float((errors ** 2).sum())
        sum_y += float(y_true.sum())
        sum_y2 += float((y_true ** 2).sum())

    predict_seconds = time.time() - start

    if n == 0:
        raise ValueError("No rows were evaluated. Check split/sample/dataset filters.")

    mae, rmse, r2 = compute_metrics_from_sums(
        n,
        sum_abs_error,
        sum_sq_error,
        sum_y,
        sum_y2
    )

    return {
        "metric_rows": int(n),
        "predict_seconds": float(predict_seconds),
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": None if r2 is None else float(r2),
        "sum_abs_error": float(sum_abs_error),
        "sum_sq_error": float(sum_sq_error),
        "sum_y": float(sum_y),
        "sum_y2": float(sum_y2),
    }

In [ ]:
def run_snowflake_xgb(
    features,
    feature_set_name,
    use_gpu,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="final",
    n_estimators=30,
    notes=""
):
    run_id = str(uuid.uuid4())

    model_scope = "global" if dataset is None else "separate"
    dataset_label = "all" if dataset is None else dataset

    task = "T10" if use_gpu else "T7"
    model_name = "Snowflake_Distributed_XGBoost_GPU" if use_gpu else "Snowflake_Distributed_XGBoost_CPU"
    compute_type = "Snowflake Ray GPU trainer" if use_gpu else "Snowflake Ray CPU trainer"

    train_sdf = make_train_sdf(
        features=features,
        dataset=dataset,
        train_sample_pct=train_sample_pct
    )

    trainer_eval_sdf = make_trainer_eval_sdf(
        features=features,
        dataset=dataset,
        trainer_eval_sample_pct=trainer_eval_sample_pct
    )

    train_rows = train_sdf.count()
    trainer_eval_rows = trainer_eval_sdf.count()

    print(f"Train rows: {train_rows:,}")
    print(f"Validation rows used by trainer: {trainer_eval_rows:,}")

    train_connector = DataConnector.from_dataframe(train_sdf)
    eval_connector = DataConnector.from_dataframe(trainer_eval_sdf)

    params = {
        "n_estimators": n_estimators,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 6,
        "learning_rate": 0.08,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    }

    estimator = XGBEstimator(
        params=params,
        scaling_config=XGBScalingConfig(use_gpu=use_gpu)
    )

    warehouse_name, warehouse_size = get_warehouse_info()
    ray_nodes, ray_resources = get_ray_metadata()

    with ResourceMonitor(interval=1.0) as monitor:
        start_train = time.time()

        booster = estimator.fit(
            dataset=train_connector,
            input_cols=features,
            label_col=TARGET_LOG,
            eval_set=eval_connector,
            verbose_eval=10
        )

        train_seconds = time.time() - start_train

    if booster is None:
        booster = estimator.get_booster()

    metric_result = evaluate_xgb_batched(
        booster=booster,
        features=features,
        split=metric_split,
        dataset=dataset,
        metric_sample_pct=metric_sample_pct
    )

    result = {
        "RUN_ID": run_id,
        "TASK": task,
        "MODEL_NAME": model_name,
        "MODEL_SCOPE": model_scope,
        "DATASET": dataset_label,
        "FEATURE_SET": feature_set_name,
        "COMPUTE_TYPE": compute_type,
        "SCALING_LABEL": scaling_label,

        "TRAIN_SPLIT": "train",
        "TRAINER_EVAL_SPLIT": "validation",
        "METRIC_SPLIT": metric_split,

        "TRAIN_SAMPLE_PCT": float(train_sample_pct),
        "TRAINER_EVAL_SAMPLE_PCT": float(trainer_eval_sample_pct),
        "METRIC_SAMPLE_PCT": float(metric_sample_pct),

        "TRAIN_ROWS": int(train_rows),
        "TRAINER_EVAL_ROWS": int(trainer_eval_rows),
        "METRIC_ROWS": int(metric_result["metric_rows"]),

        "N_ESTIMATORS": int(n_estimators),
        "TRAIN_SECONDS": float(train_seconds),
        "PREDICT_SECONDS": float(metric_result["predict_seconds"]),

        "MAE": float(metric_result["mae"]),
        "RMSE": float(metric_result["rmse"]),
        "R2": None if metric_result["r2"] is None else float(metric_result["r2"]),

        "SUM_ABS_ERROR": float(metric_result["sum_abs_error"]),
        "SUM_SQ_ERROR": float(metric_result["sum_sq_error"]),
        "SUM_Y": float(metric_result["sum_y"]),
        "SUM_Y2": float(metric_result["sum_y2"]),

        "PEAK_PROCESS_MEMORY_MB": float(monitor.peak_process_memory_mb),
        "PEAK_GPU_MEMORY_MB": float(monitor.peak_gpu_memory_mb),

        "RAY_NODES": int(ray_nodes) if ray_nodes is not None else None,
        "RAY_RESOURCES": ray_resources,

        "WAREHOUSE_NAME": warehouse_name,
        "WAREHOUSE_SIZE": warehouse_size,

        "NOTES": notes,
    }

    save_xgb_result(result)
    # save_feature_importance(run_id, model_name, feature_set_name, booster)

    print("\nSaved result:")
    print(result)

    print("\nFeature importance:")
    print(booster.get_score(importance_type="gain"))

    return booster, result

In [ ]:
def save_combined_separate_result(results, scaling_label, feature_set_name, use_gpu=True, notes=""):
    total_rows = sum(r["METRIC_ROWS"] for r in results)
    total_abs = sum(r["SUM_ABS_ERROR"] for r in results)
    total_sq = sum(r["SUM_SQ_ERROR"] for r in results)
    total_y = sum(r["SUM_Y"] for r in results)
    total_y2 = sum(r["SUM_Y2"] for r in results)

    mae, rmse, r2 = compute_metrics_from_sums(
        total_rows,
        total_abs,
        total_sq,
        total_y,
        total_y2
    )

    train_rows = sum(r["TRAIN_ROWS"] for r in results)
    trainer_eval_rows = sum(r["TRAINER_EVAL_ROWS"] for r in results)
    train_seconds = sum(r["TRAIN_SECONDS"] for r in results)
    predict_seconds = sum(r["PREDICT_SECONDS"] for r in results)

    warehouse_name, warehouse_size = get_warehouse_info()
    ray_nodes, ray_resources = get_ray_metadata()

    result = {
        "RUN_ID": str(uuid.uuid4()),
        "TASK": "T10" if use_gpu else "T7",
        "MODEL_NAME": "Combined_Separate_XGBoost_Models",
        "MODEL_SCOPE": "separate_combined",
        "DATASET": "yellow+green+fhv+fhvhv",
        "FEATURE_SET": feature_set_name,
        "COMPUTE_TYPE": "Combined metrics from four separate XGBoost models",
        "SCALING_LABEL": scaling_label,

        "TRAIN_SPLIT": "train",
        "TRAINER_EVAL_SPLIT": "validation",
        "METRIC_SPLIT": "test",

        "TRAIN_SAMPLE_PCT": results[0]["TRAIN_SAMPLE_PCT"],
        "TRAINER_EVAL_SAMPLE_PCT": results[0]["TRAINER_EVAL_SAMPLE_PCT"],
        "METRIC_SAMPLE_PCT": results[0]["METRIC_SAMPLE_PCT"],

        "TRAIN_ROWS": int(train_rows),
        "TRAINER_EVAL_ROWS": int(trainer_eval_rows),
        "METRIC_ROWS": int(total_rows),

        "N_ESTIMATORS": results[0]["N_ESTIMATORS"],
        "TRAIN_SECONDS": float(train_seconds),
        "PREDICT_SECONDS": float(predict_seconds),

        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": None if r2 is None else float(r2),

        "SUM_ABS_ERROR": float(total_abs),
        "SUM_SQ_ERROR": float(total_sq),
        "SUM_Y": float(total_y),
        "SUM_Y2": float(total_y2),

        "PEAK_PROCESS_MEMORY_MB": max(r["PEAK_PROCESS_MEMORY_MB"] for r in results),
        "PEAK_GPU_MEMORY_MB": max(r["PEAK_GPU_MEMORY_MB"] for r in results),

        "RAY_NODES": int(ray_nodes) if ray_nodes is not None else None,
        "RAY_RESOURCES": ray_resources,

        "WAREHOUSE_NAME": warehouse_name,
        "WAREHOUSE_SIZE": warehouse_size,

        "NOTES": notes,
    }

    save_xgb_result(result)

    print("\nSaved combined separate-model result:")
    print(result)

    return result

In [ ]:
scale_ray_cluster_to(1)

In [ ]:
booster_base_1node, result_base_1node = run_snowflake_xgb(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_baseline_1_node_full_train_full_test",
    n_estimators=30,
    notes="Baseline XGBoost GPU run without T5 augmentation. Full train and full test. 1 Ray node."
)

In [ ]:
scale_ray_cluster_to(2)

In [ ]:
booster_base_2nodes, result_base_2nodes = run_snowflake_xgb(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_baseline_2_nodes_full_train_full_test",
    n_estimators=30,
    notes="Baseline XGBoost GPU run without T5 augmentation. Full train and full test. 2 Ray nodes."
)

In [ ]:
scale_ray_cluster_to(4)

In [ ]:
booster_base_4nodes, result_base_4nodes = run_snowflake_xgb(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_baseline_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Baseline XGBoost GPU run without T5 augmentation. Full train and full test. 4 Ray nodes."
)

In [ ]:
scale_ray_cluster_to(2)

In [ ]:
booster_aug_2nodes, result_aug_2nodes = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_augmented_2_nodes_full_train_full_test",
    n_estimators=30,
    notes="Augmented XGBoost GPU run with T5 features. Full train and full test. 2 Ray nodes."
)

In [ ]:
scale_ray_cluster_to(4)

In [ ]:
booster_aug_2nodes, result_aug_2nodes = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_augmented_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Augmented XGBoost GPU run with T5 features. Full train and full test. 4 Ray nodes."
)

In [ ]:
scale_ray_cluster_to(4)

In [ ]:
separate_results = []

In [ ]:
booster_yellow, result_yellow = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset="yellow",
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_separate_yellow_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Separate Yellow XGBoost GPU model with T5 features. Full train and full test. 4 Ray nodes."
)
separate_results.append(result_yellow)

In [ ]:
booster_green, result_green = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset="green",
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_separate_green_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Separate Green XGBoost GPU model with T5 features. Full train and full test. 4 Ray nodes."
)
separate_results.append(result_green)

In [ ]:
booster_fhv, result_fhv = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset="fhv",
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_separate_fhv_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Separate FHV XGBoost GPU model with T5 features. Full train and full test. 4 Ray nodes."
)
separate_results.append(result_fhv)

In [ ]:
booster_fhvhv, result_fhvhv = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset="fhvhv",
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_separate_fhvhv_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Separate FHVHV XGBoost GPU model with T5 features. Full train and full test. 4 Ray nodes."
)
separate_results.append(result_fhvhv)

In [ ]:
combined_separate_result = save_combined_separate_result(
    separate_results,
    scaling_label="xgb_gpu_combined_separate_4_nodes_full_train_full_test",
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    notes="Combined metrics from four separate dataset-specific XGBoost GPU models."
)

In [ ]:
SELECT
    task,
    model_name,
    model_scope,
    dataset,
    feature_set,
    scaling_label,
    train_rows,
    metric_rows,
    ray_nodes,
    n_estimators,
    train_seconds,
    predict_seconds,
    mae,
    rmse,
    r2,
    peak_process_memory_mb,
    peak_gpu_memory_mb,
    created_at
FROM GOLD.T7_T10_XGB_RESULTS
ORDER BY created_at;

In [ ]:
scale_ray_cluster_to(4)

In [ ]:
booster_aug_cpu_4nodes, result_aug_cpu_4nodes = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=False,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_cpu_augmented_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="CPU distributed XGBoost run with T5 augmentation. Full train and full test. 4 Ray CPU nodes."
)

In [ ]:
%%sql -r dataframe_4
SELECT
    task,
    model_name,
    model_scope,
    dataset,
    feature_set,
    scaling_label,
    train_rows,
    metric_rows,
    ray_nodes,
    n_estimators,
    train_seconds,
    predict_seconds,
    mae,
    rmse,
    r2,
    peak_process_memory_mb,
    peak_gpu_memory_mb,
    created_at
FROM GOLD.T7_T10_XGB_RESULTS
ORDER BY created_at;